# TP3 â€” Agentic Patterns & Multi-Agent Systems with CrewAI
## Module 2IAiasd 9.7 â€” Agentic AI Â· IMT Mines AlÃ¨s

**Duration:** ~1 hour (first half of the session â€” TP4 LangGraph follows)
**Prerequisites:**
- TP1 completed â€” you know the **ReAct loop** (reason â†’ act â†’ observe)
- TP2 completed â€” the `chroma_db/` folder (index of your research paper) is in the same directory as this notebook
- `ollama serve` is running, `qwen2.5:3b` and `nomic-embed-text` are pulled

**Objective:** Go beyond ReAct. Implement two other agentic patterns from CM1 â€” **reflection** and **planning** â€”
by hand, then build a **multi-agent system** with CrewAI where a *researcher* and a *writer* collaborate
on your research paper.

---

### Structure

| # | Section | Concepts | Time |
|---|---------|----------|------|
| 1 | Reflection | generate â†’ critique â†’ revise, structured critic | 15 min |
| 2 | Planning | Plan-and-Execute, Tree of Thoughts (ToT) | 15 min |
| 3 | CrewAI | agents with role / goal / backstory, tasks, crew â€” researcher + writer | 30 min |

### What you will build today

```
  Section 1                Section 2                       Section 3
  â”€â”€â”€â”€â”€â”€â”€â”€â”€                â”€â”€â”€â”€â”€â”€â”€â”€â”€                       â”€â”€â”€â”€â”€â”€â”€â”€â”€
  â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”              goal â”€â”€â–º PLANNER â”€â”€â–º steps       inputs {topic}
  â”‚GENERATORâ”‚â—„â”€â”€â”                       â”‚                        â”‚
  â””â”€â”€â”€â”€â”¬â”€â”€â”€â”€â”˜   â”‚ revise                â–¼                   â”Œâ”€â”€â”€â”€â–¼â”€â”€â”€â”€â”€â”€â”  search_report
       â–¼        â”‚              EXECUTOR (step by step)      â”‚RESEARCHER â”‚â—„â”€â”€ (ChromaDB TP2)
  â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”   â”‚                                           â””â”€â”€â”€â”€â”¬â”€â”€â”€â”€â”€â”€â”˜
  â”‚ CRITIC  â”‚â”€â”€â”€â”˜           problem â”€â”€â–º k ideas â”€â”€â–º scores       â–¼ notes
  â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜                               â””â”€â”€â–º best â”€â”€â–º   â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
                                                 develop    â”‚  WRITER   â”‚â”€â”€â–º summary.md
                                                            â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
```

> ðŸ“– **Reference:** CM1 â€” the four agentic patterns (reflection, tool use, planning, multi-agent).
> In TP1 you implemented *tool use* (ReAct). Today: the three others.

---
## Setup

In [1]:
%pip install -q crewai langchain-ollama langchain-chroma chromadb pydantic

print("âœ… Dependencies installed")

Note: you may need to restart the kernel to use updated packages.
âœ… Dependencies installed


In [1]:
import os, textwrap, requests
from pathlib import Path
from pydantic import BaseModel, Field

# CrewAI: disable telemetry and the interactive "view traces?" prompt
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"]        = "true"
os.environ["CREWAI_TRACING_ENABLED"]   = "false"

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

MODEL       = "qwen2.5:3b"
CREW_MODEL  = "qwen2.5:3b"   # ðŸ‘‰ use "qwen2.5:7b" if your laptop has â‰¥ 16 GB RAM (much more reliable with CrewAI)
EMBED_MODEL = "nomic-embed-text"
CHROMA_DIR  = "./chroma_db"  # built in TP2 Part 1

# Health check
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"âœ… Ollama running | available models: {models}")
except Exception:
    print("âŒ Ollama not reachable â€” run: ollama serve")

llm          = ChatOllama(model=MODEL, temperature=0)     # deterministic: critic, planner, evaluator
llm_creative = ChatOllama(model=MODEL, temperature=0.7)   # diverse: generator, idea generation


def ask(system: str, user: str, model=llm) -> str:
    """One LLM call with a system prompt and a user message."""
    return model.invoke([SystemMessage(system), HumanMessage(user)]).content


def show(title: str, text: str, width: int = 78) -> None:
    print(f"\nâ”Œâ”€ {title} " + "â”€" * max(0, width - len(title) - 4))
    for line in str(text).splitlines() or [""]:
        print("â”‚ " + "\nâ”‚ ".join(textwrap.wrap(line, width - 2) or [""]))
    print("â””" + "â”€" * (width - 1))


print(f"âœ… LLM ready: {MODEL}")

âœ… Ollama running | available models: ['nomic-embed-text:latest', 'qwen2.5:3b']
âœ… LLM ready: qwen2.5:3b


---
## Section 1 â€” Reflection: generate â†’ critique â†’ revise

In ReAct, the agent improves its answer by **acting on the world** (tools).
In **Reflection**, the agent improves its answer by **criticising itself**:

```
  task â”€â”€â–º GENERATOR â”€â”€â–º draft â”€â”€â–º CRITIC â”€â”€â–º feedback
               â–²                                  â”‚
               â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€ REVISE â—„â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
                     (until APPROVED or max_rounds)
```

The **same LLM** plays two roles with two different system prompts.
This works because evaluating a text is easier than writing it â€” the critic sees the draft "from outside".

> ðŸ“š Self-Refine (Madaan et al., 2023) Â· Reflexion (Shinn et al., 2023)

### 1.1 â€” Describe your research paper

In [2]:
# 👉 Describe YOUR experience in 2–3 sentences (institution, mission, main result)
# Context: undergraduate research project (Iniciação Científica) that led to a paper
# where I am the first author. Later cells reuse MY_PAPER as context.
MY_PAPER = """
Undergraduate research project (Iniciação Científica) at the Institute of Computing,
University of Campinas (Unicamp), Campinas, Brazil, which resulted in a paper where I am
the first author: "How Well Do Small Language Models Understand the Brazilian Educational
Context? An Evaluation on ENEM, FUVEST and Unicamp".
Mission: build a zero-shot benchmark of 12 language models (6 Small Language Models with
at most 14B parameters and 6 larger reference LLMs) on 3,701 Portuguese questions from
Brazil's main university entrance exams (ENEM, FUVEST and Unicamp), replacing image-based
questions with text descriptions and analysing the cost-accuracy trade-off.
Main result: the compact Qwen3.5-4B reached 84.5% average accuracy, beating the much larger
Gemma-3-27B (76.7%) and coming within 8 points of the best LLM (GLM-5.1, 92.0%) at a
fraction of the inference cost, while showing that small models still struggle with
mathematics, formal reasoning and strict output formats.
"""

TASK = (
    "Write a LinkedIn post of at most 80 words announcing the end of this undergraduate "
    "research project and the publication of the resulting paper, of which I am the first author. "
    "Professional tone, no more than 2 emojis, end with a thank-you to my co-authors and advisors.\n\n"
    f"Research project: {MY_PAPER}"
)
print(TASK)

Write a LinkedIn post of at most 80 words announcing the end of this undergraduate research project and the publication of the resulting paper, of which I am the first author. Professional tone, no more than 2 emojis, end with a thank-you to my co-authors and advisors.

Research project: 
Undergraduate research project (Iniciação Científica) at the Institute of Computing,
University of Campinas (Unicamp), Campinas, Brazil, which resulted in a paper where I am
the first author: "How Well Do Small Language Models Understand the Brazilian Educational
Context? An Evaluation on ENEM, FUVEST and Unicamp".
Mission: build a zero-shot benchmark of 12 language models (6 Small Language Models with
at most 14B parameters and 6 larger reference LLMs) on 3,701 Portuguese questions from
Brazil's main university entrance exams (ENEM, FUVEST and Unicamp), replacing image-based
questions with text descriptions and analysing the cost-accuracy trade-off.
Main result: the compact Qwen3.5-4B reached 84.5% a

### 1.2 â€” The three roles

In [3]:
GENERATOR_SYS = "You are a professional copywriter. Follow every constraint of the task exactly. Return only the text."

CRITIC_SYS = """You are a demanding editor. Review the DRAFT against the TASK.
List at most 3 concrete problems (length, tone, missing facts, cliches, emojis).
If the draft fully respects the task, answer with exactly one word: APPROVED"""


def generate(task: str) -> str:
    return ask(GENERATOR_SYS, task, llm_creative)


def critique(task: str, draft: str) -> str:
    return ask(CRITIC_SYS, f"TASK:\n{task}\n\nDRAFT:\n{draft}")


def revise(task: str, draft: str, feedback: str) -> str:
    return ask(
        GENERATOR_SYS,
        f"TASK:\n{task}\n\nPREVIOUS DRAFT:\n{draft}\n\nEDITOR FEEDBACK:\n{feedback}\n\n"
        "Rewrite the draft so that every point of the feedback is fixed. Return only the new draft.",
        llm_creative,
    )


print("âœ… generate / critique / revise ready")

âœ… generate / critique / revise ready


### 1.3 â€” The reflection loop

In [4]:
def reflection_loop(task: str, max_rounds: int = 3, verbose: bool = True):
    draft = generate(task)
    history = [draft]
    if verbose:
        show(f"DRAFT 0  ({len(draft.split())} words)", draft)

    for r in range(1, max_rounds + 1):
        feedback = critique(task, draft)
        if verbose:
            show(f"CRITIC â€” round {r}", feedback)
        if "APPROVED" in feedback.upper():
            break
        draft = revise(task, draft, feedback)
        history.append(draft)
        if verbose:
            show(f"DRAFT {r}  ({len(draft.split())} words)", draft)

    return draft, history


final_post, history = reflection_loop(TASK)

print("\nWord count per draft:", [len(d.split()) for d in history])

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 0  (56 words) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Congratulations on the successful completion of our undergraduate research
â”‚ project and the publication of our paper! 📜✨ As the first author, I'm
â”‚ honored to share this achievement with my co-authors and advisors. Our
â”‚ research evaluated small language models in understanding the Brazilian
â”‚ educational context, achieving impressive results. Thank you for your
â”‚ support and collaboration. 🙏🚀
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

â”Œâ”€ CRITIC â€” round 1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 1  (56 words) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Congratulations on the successful completion of our undergraduate research
â”‚ project and the publication of our paper! 📜✨ As the first author, I'm
â”‚ honored to share this achievement with my co-authors and advisors. Our
â”‚ research evaluated small language models in understanding the Brazilian
â”‚ educational context, achieving impressive results. Thank you for your
â”‚ support and collaboration. 🙏🚀
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

â”Œâ”€ CRITIC â€” round 2 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 2  (102 words) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Congratulations on the successful completion of our undergraduate research
â”‚ project and the publication of our paper! 📜✨ As the first author, I'm
â”‚ honored to share this achievement with my co-authors and advisors. Our
â”‚ research evaluated small language models in understanding the Brazilian
â”‚ educational context, achieving impressive results. The compact Qwen3.5-4B
â”‚ reached 84.5% average accuracy, beating the much larger Gemma-3-27B (76.7%)
â”‚ and coming within 8 points of the best LLM (GLM-5.1, 92.0%) at a fraction of
â”‚ the inference cost, while showing that small models still struggle with
â”‚ mathematics, formal reasoning, and strict output formats. Thank you for your
â”‚ support and collaboration. 🙏🚀
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 3  (98 words) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Congratulations on the successful completion of our undergraduate research
â”‚ project and the publication of our paper! 📜✨ As the first author, I'm
â”‚ honored to share this achievement with my co-authors and advisors. Our
â”‚ research evaluated small language models in understanding the Brazilian
â”‚ educational context, achieving impressive results. The compact Qwen3.5-4B
â”‚ reached 84.5% average accuracy, beating the much larger Gemma-3-27B (76.7%)
â”‚ and coming within 8 points of the best LLM (GLM-5.1, 92.0%) at a fraction of
â”‚ the inference cost, while still struggling with mathematics, formal
â”‚ reasoning, and strict output formats. Thank you for your support and
â”‚ collaboration. 🙏🚀
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â

**Observe:**
- Did the word count move towards the 80-word limit?
- Did the critic ever say `APPROVED`? Small models are often either **too lenient** (approve immediately)
  or **never satisfied** (always find something). Both make the free-text stopping rule unreliable.

### 1.4 âœï¸ Exercise â€” A structured critic with a score

Replace the free-text critic with a **structured output** (CM2): the critic returns a score and a list of issues,
and the loop stops when `score >= threshold`.

**Hint:** some checks are better done **by code** than by the LLM. The word count is one of them.

In [ ]:
import re

from langchain_core.exceptions import OutputParserException


class Critique(BaseModel):
    """Structured verdict returned by the critic LLM.

    The ``Field`` constraints are enforced by Pydantic when the JSON answer is
    parsed (``ge``/``le`` -> ``minimum``/``maximum`` and ``max_length`` ->
    ``maxItems`` in the JSON Schema sent to Ollama).
    """

    score: int = Field(ge=1, le=10, description="Overall quality of the draft, from 1 (bad) to 10 (excellent)")
    issues: list[str] = Field(max_length=3, description="At most 3 concrete problems to fix. Empty list if none.")


structured_critic = llm.with_structured_output(Critique)

STRUCTURED_CRITIC_SYS = (
    "You are a demanding editor. Review the DRAFT against the TASK. "
    "Give a score from 1 to 10 and list at most 3 concrete problems to fix "
    "(tone, missing or wrong facts, clichés). Return an empty list only if the draft is excellent."
)

# Emoji ranges: pictographs/symbols (U+1F300–U+1FAFF) and misc symbols/dingbats (U+2600–U+27BF).
EMOJI_PATTERN = re.compile("[\U0001F300-\U0001FAFF☀-➿]")


def count_emojis(text: str) -> int:
    """Count the emoji characters in a text.

    Args:
        text: The text to inspect.

    Returns:
        Number of characters that fall in the usual emoji Unicode ranges.
    """
    return len(EMOJI_PATTERN.findall(text))


def critique_structured(task: str, draft: str) -> Critique:
    """Ask the critic LLM for a structured verdict on a draft.

    Args:
        task: The original writing task, with all its constraints.
        draft: The draft to review.

    Returns:
        The parsed ``Critique``. If the LLM answer violates the schema
        constraints (e.g. 4 issues or a score of 11), a minimal failing
        ``Critique`` is returned instead of crashing the loop.
    """
    try:
        return structured_critic.invoke([
            SystemMessage(STRUCTURED_CRITIC_SYS),
            HumanMessage(f"TASK:\n{task}\n\nDRAFT:\n{draft}"),
        ])
    except OutputParserException as err:
        print(f"⚠️  Critic answer rejected by the schema: {str(err)[:120]}...")
        return Critique(score=1, issues=["Improve the draft so that it respects every constraint of the task."])


def reflection_loop_v2(
    task: str,
    threshold: int = 8,
    max_words: int = 80,
    max_emojis: int = 2,
    max_rounds: int = 3,
    verbose: bool = True,
) -> tuple[str, list[int], list[str]]:
    """Reflection loop driven by a structured critic plus deterministic code checks.

    Each round: the LLM critic returns a score and issues; the code counts words
    and emojis and adds its own issues; the loop stops when the score reaches
    ``threshold`` AND every code check passes, otherwise the draft is revised.

    Args:
        task: The writing task given to the generator.
        threshold: Minimum critic score (1–10) required to stop.
        max_words: Maximum number of words allowed (checked by code).
        max_emojis: Maximum number of emojis allowed (checked by code).
        max_rounds: Maximum number of critique/revise rounds.
        verbose: If True, print every draft, score and list of issues.

    Returns:
        A tuple ``(final_draft, scores, history)``: the last draft, the critic
        score of each round, and every draft produced (draft 0 first).
    """
    draft = generate(task)
    history = [draft]
    scores: list[int] = []
    if verbose:
        show(f"DRAFT 0  ({len(draft.split())} words, {count_emojis(draft)} emojis)", draft)

    for r in range(1, max_rounds + 1):
        # 1. LLM judgement (format guaranteed by the schema)
        c = critique_structured(task, draft)
        scores.append(c.score)
        issues = list(c.issues)

        # 2. Deterministic checks done by code
        n_words, n_emojis = len(draft.split()), count_emojis(draft)
        if n_words > max_words:
            issues.append(f"Too long: {n_words} words, the limit is {max_words}. Shorten it.")
        if n_emojis > max_emojis:
            issues.append(f"Too many emojis: {n_emojis}, the limit is {max_emojis}. Remove some.")

        if verbose:
            show(f"CRITIC — round {r}  (score {c.score}/10)", "\n".join(f"- {i}" for i in issues) or "(no issues)")

        # 3. Stopping rule: numeric and decided by code
        if c.score >= threshold and n_words <= max_words and n_emojis <= max_emojis:
            if verbose:
                print(f"\n✅ Stop at round {r}: score {c.score} ≥ {threshold} and all code checks pass.")
            break

        # 4. Revise with actionable feedback (fallback if the list is empty)
        feedback = "\n".join(issues) or "Improve the overall quality of the post."
        draft = revise(task, draft, feedback)
        history.append(draft)
        if verbose:
            show(f"DRAFT {r}  ({len(draft.split())} words, {count_emojis(draft)} emojis)", draft)

    return draft, scores, history


final_v2, scores, history = reflection_loop_v2(TASK)

# ── Summary ──────────────────────────────────────────────────────────────────
words_per_draft = [len(d.split()) for d in history]
emojis_per_draft = [count_emojis(d) for d in history]
final_ok = words_per_draft[-1] <= 80 and emojis_per_draft[-1] <= 2

print("\n" + "═" * 60)
print("Scores per round      :", scores)
print("Words per draft       :", words_per_draft)
print("Emojis per draft      :", emojis_per_draft)
print("Drafts produced       :", len(history))
print("LLM calls made        :", 1 + len(scores) + (len(history) - 1), "(1 generate + critiques + revisions)")
print("Final draft passes the code checks (≤ 80 words, ≤ 2 emojis)?", final_ok)
print("═" * 60)
show("FINAL POST", final_v2)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 0  (53 words, 3 emojis) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Undergraduate research project concludes successfully, with our paper now
â”‚ published! "How Well Do Small Language Models Understand the Brazilian
â”‚ Educational Context?" is our first-authored contribution, evaluating 12
â”‚ models on key exams. Compact Qwen3.5-4B outperformed larger models in cost
â”‚ efficiency. Thank you to my co-authors and advisors for this incredible
â”‚ journey! 📝💻 #ResearchSuccess
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ CRITIC — round 1  (score 6/10) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ - The draft exceeds the 80-word limit.
â”‚ - The emoji count is correct, but the draft could benefit from a more
â”‚ concise summary of the research.
â”‚ - The draft could include a clearer mention of the co-authors and advisors
â”‚ to ensure proper attribution.
â”‚ - Too many emojis: 3, the limit is 2. Remove some.
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 1  (63 words, 3 emojis) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Undergraduate research project concludes successfully, with our paper now
â”‚ published! Evaluating 12 language models on key Brazilian university
â”‚ entrance exams (ENEM, FUVEST, and Unicamp), Compact Qwen3.5-4B outperformed
â”‚ larger models in cost efficiency. It reached 84.5% average accuracy, beating
â”‚ Gemma-3-27B and coming within 8 points of the best LLM (GLM-5.1). Thank you
â”‚ to my co-authors and advisors for this incredible journey! 📝💻
â”‚ #ResearchSuccess
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


... (5 outputs left out)


â”Œâ”€ CRITIC — round 3  (score 7/10) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ - The draft exceeds the 80-word limit.
â”‚ - The emoji count is correct, but the post could benefit from a more concise
â”‚ summary of the main result.
â”‚ - The draft does not mention the co-authors and advisors, which is a crucial
â”‚ part of the task.
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ DRAFT 3  (83 words, 1 emojis) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ 🎉 Undergraduate research project concludes successfully, with our paper now
â”‚ published! Evaluating 12 language models on key Brazilian university
â”‚ entrance exams (ENEM, FUVEST, and Unicamp), Compact Qwen3.5-4B outperformed
â”‚ larger models in cost efficiency, reaching 84.5% average accuracy, just 8
â”‚ points away from the best LLM (GLM-5.1). This outperformed Gemma-3-27B.
â”‚ Thank you to my co-authors and advisors for this incredible journey!
â”‚ #ResearchSuccess
â”‚ 
â”‚ Main result: Compact Qwen3.5-4B achieved 84.5% accuracy, beating larger
â”‚ models like Gemma-3-27B and coming close to GLM-5.1, while significantly
â”‚ reducing inference cost.
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


**Questions:**
1. Which checks should be done by **code** rather than by the LLM critic? Why?
   > *Your answer:* Every check that can be measured exactly from the text: the word count, the emoji count and, more generally, any hard constraint of the task. Even simple factual checks can be done by code, e.g. verifying that every number in the post appears in the source context (it would have caught the hallucinated "12,123 questions" of Section 1.3). The stopping rule itself (`score >= threshold`) must also be code. Code is deterministic, free and instant, while an LLM is slower, costs a call and is unreliable at counting, because it sees tokens, not words or characters (in Section 1.3 the critic never noticed the 106 words). The LLM critic should be kept for what code cannot judge: tone, clarity and relevance.
2. How many LLM calls does a reflection loop with `r` rounds cost (worst case)?
   > *Your answer:* **2r + 1** calls: 1 call to generate the first draft, then, in each of the `r` rounds, 1 call to the critic and 1 call to revise the draft (worst case = the loop never stops early). The structured output adds no call (the JSON format is enforced inside the same call) and the code checks cost 0 LLM calls. Best case: 2 calls (draft approved in round 1). In our run, r = 3 → 7 calls.

---
## Section 2 â€” Planning: Plan-and-Execute and Tree of Thoughts

ReAct **interleaves** thinking and acting: it decides one step at a time.
Planning patterns **think ahead** before acting.

| Pattern | Idea | Strength | Weakness |
|---------|------|----------|----------|
| **ReAct** (TP1) | think â†’ act â†’ observe, one step at a time | adapts to each observation | myopic, can loop |
| **Plan-and-Execute** | write the whole plan first, then execute each step | global vision, fewer "lost" steps | plan can be wrong and is not revised |
| **Tree of Thoughts** | explore several reasoning branches, score them, keep the best | avoids committing to the first idea | many LLM calls |

### 2.1 â€” Plan-and-Execute

```
  goal â”€â”€â–º PLANNER â”€â”€â–º [step 1, step 2, step 3, ...]
                              â”‚
                              â–¼
                   EXECUTOR runs step 1 â”€â”€â–º result 1 â”€â”
                   EXECUTOR runs step 2 (sees result 1) â”€â”€â–º result 2 â”€â”
                   ...                                                 â–¼
                                                                 final results
```

In [6]:
class Plan(BaseModel):
    steps: list[str] = Field(description="3 to 5 short, concrete, ordered steps")


planner = llm.with_structured_output(Plan)

GOAL = (
    "Prepare a 5-minute oral presentation of my first-author research paper on Small Language "
    "Models for Brazilian entrance exams, for a jury of engineering teachers."
)

plan = planner.invoke([
    SystemMessage("You are a planning assistant. Break the goal into 3 to 5 concrete, ordered steps. "
                  "Do NOT execute the steps."),
    HumanMessage(f"Goal: {GOAL}\nContext: {MY_PAPER}"),
])

print("ðŸ“‹ PLAN")
for i, step in enumerate(plan.steps, 1):
    print(f"  {i}. {step}")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


ðŸ“‹ PLAN
  1. Step 1: Research and Organize Key Points
  2. Step 2: Draft an Outline and Script
  3. Step 3: Practice and Refine Delivery
  4. Step 4: Prepare Visual Aids and Materials
  5. Step 5: Final Review and Presentation


In [7]:
def execute_plan(goal: str, plan: Plan, context: str) -> list[str]:
    results = []
    for i, step in enumerate(plan.steps, 1):
        previous = "\n".join(f"Step {j} result: {r[:300]}" for j, r in enumerate(results, 1))
        out = ask(
            "You execute ONE step of a plan. Be concrete and concise (max 80 words).",
            f"Goal: {goal}\nContext: {context}\n\nResults so far:\n{previous or '(none)'}\n\n"
            f"Current step: {step}",
        )
        results.append(out)
        show(f"STEP {i} â€” {step[:50]}", out)
    return results


step_results = execute_plan(GOAL, plan, MY_PAPER)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ STEP 1 â€” Step 1: Research and Organize Key Points â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ Step 1: Research and Organize Key Points: Identify main findings, including
â”‚ Qwen3.5-4B's performance, cost-effectiveness, and challenges in specific
â”‚ domains. Outline structure: introduction, methodology, results, discussion,
â”‚ conclusion.
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

â”Œâ”€ STEP 2 â€” Step 2: Draft an Outline and Script â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ Step 2: Drafted an outline and script for the presentation, including
â”‚ introduction, methodology, results, discussion, and conclusion, focusing on
â”‚ Qwen3.5-4B's performance and cost-effectiveness.
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”

> ðŸ’¡ **Re-planning.** A more robust version adds a **re-planner** after each step:
> *"given what we learned, should the remaining steps change?"*. This is a loop with a decision â€”
> exactly what LangGraph (TP4) is designed to express.

### 2.2 â€” Tree of Thoughts (ToT)

```
                         problem
             â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
          idea A         idea B         idea C        â† generate k thoughts   (breadth)
       feas 6 / imp 5  feas 8 / imp 9  feas 4 / imp 7 â† evaluate each thought
                            â”‚
                       develop idea B                  â† expand the best branch (depth)
```

> ðŸ“š Yao et al., *Tree of Thoughts: Deliberate Problem Solving with LLMs*, 2023.

The problem below links to your **project**: find a first AI-agent use case that turns the findings of your paper into a real application for Brazilian education.

In [ ]:
PROBLEM = (
    "Propose ONE first use case of an LLM-based AI agent, built on a Small Language Model and "
    "grounded in the findings of my paper, to support students preparing for Brazilian university "
    "entrance exams (ENEM, FUVEST, Unicamp) in resource-constrained public schools. "
    "It must take into account the weaknesses reported in the paper (mathematics, formal "
    "reasoning, strict output formats). "
    f"Paper context: {MY_PAPER}"
)


class Evaluation(BaseModel):
    feasibility: int = Field(description="Feasibility in 3 months by 2 engineering students, from 1 to 10")
    impact: int = Field(description="Educational impact for Brazilian students and public schools, from 1 to 10")
    justification: str = Field(description="One sentence justifying the two scores")


evaluator = llm.with_structured_output(Evaluation)


def generate_thoughts(problem: str, k: int = 3) -> list[str]:
    """Generate k ideas sequentially, asking each time for an idea different from the previous ones."""
    thoughts = []
    for _ in range(k):
        already = "\n".join(f"- {t}" for t in thoughts) or "(none)"
        idea = ask(
            "Propose ONE idea in at most 2 sentences. It must be clearly different from the ideas already listed.",
            f"Problem: {problem}\n\nIdeas already proposed:\n{already}",
            llm_creative,
        )
        thoughts.append(idea.strip())
    return thoughts


def evaluate_thought(problem: str, thought: str) -> Evaluation:
    return evaluator.invoke([
        SystemMessage("You are a strict technical reviewer. Score the idea honestly."),
        HumanMessage(f"Problem: {problem}\n\nIdea: {thought}"),
    ])


def tree_of_thoughts(problem: str, k: int = 3):
    thoughts = generate_thoughts(problem, k)
    scored = []
    for i, t in enumerate(thoughts, 1):
        e = evaluate_thought(problem, t)
        total = e.feasibility + e.impact
        scored.append((total, t, e))
        show(f"IDEA {i} â€” feasibility {e.feasibility} Â· impact {e.impact} Â· total {total}",
             f"{t}\n\nâ†’ {e.justification}")

    best_total, best_idea, _ = max(scored, key=lambda x: x[0])
    development = ask(
        "Develop this idea into a short action plan: 4 bullet points (data, tools, agent pattern, success metric).",
        f"Problem: {problem}\n\nSelected idea: {best_idea}",
    )
    show(f"ðŸ† BEST IDEA (total {best_total}) â€” DEVELOPED", development)
    return best_idea, development


best_idea, development = tree_of_thoughts(PROBLEM, k=3)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP


â”Œâ”€ IDEA 1 â€” feasibility 7 Â· impact 8 Â· total 15 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ This AI agent can help students by generating detailed explanations and
â”‚ alternative problem-solving strategies for mathematics and formal reasoning
â”‚ questions found in the ENEM, FUVEST, and Unicamp exams. It can also provide
â”‚ adaptive feedback, correcting minor errors and guiding students through
â”‚ understanding strict output formats, all while operating within the resource
â”‚ constraints of public school computers.
â”‚ 
â”‚ â†’ The idea is feasible given the results from the paper, which show that a
â”‚ small language model can achieve high accuracy on the ENEM, FUVEST, and
â”‚ Unicamp exams. The idea of generating detailed explanations and alternative
â”‚ problem-solving strategies for mathematics and formal reasoning questions is
â”‚ impactful as it directly addresses the weaknesses identified in the paper.
â”‚ However, the idea cou

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ IDEA 2 â€” feasibility 7 Â· impact 8 Â· total 15 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ The AI agent can serve as a personalized study companion for students,
â”‚ providing tailored explanations and examples for difficult mathematics and
â”‚ formal reasoning problems, and adapting to the specific challenges faced by
â”‚ students in strict output format exams, all while minimizing resource usage
â”‚ to suit the limited capabilities of public school computers.
â”‚ 
â”‚ â†’ The idea is feasible given the results from the paper, which show that
â”‚ small language models can perform well on the ENEM, FUVEST, and Unicamp
â”‚ exams, even surpassing larger models in terms of cost-effectiveness. The
â”‚ idea of using the AI agent as a personalized study companion for students is
â”‚ impactful, as it addresses the specific weaknesses identified in the paper
â”‚ (mathematics, formal reasoning, strict output formats) and the resource
â”‚ constrai

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ IDEA 3 â€” feasibility 7 Â· impact 8 Â· total 15 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ The AI agent can generate detailed, student-focused explanations for
â”‚ mathematics and formal reasoning problems, while also offering practice
â”‚ exercises and interactive quizzes that adjust to the student's proficiency
â”‚ level, thereby addressing the weaknesses identified in the paper and
â”‚ supporting students in resource-constrained public school environments. It
â”‚ can also simulate strict output format exams, providing immediate feedback
â”‚ and explanations for each question to help students prepare effectively.
â”‚ 
â”‚ â†’ The idea is feasible given the results from the paper, which indicate
â”‚ that small language models can achieve high accuracy on the ENEM, FUVEST,
â”‚ and Unicamp exams. The proposed use case addresses the identified weaknesses
â”‚ in mathematics and formal reasoning, and the need for practice exercises and
â”‚ i

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



â”Œâ”€ ðŸ† BEST IDEA (total 15) â€” DEVELOPED â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ ### Action Plan
â”‚ 
â”‚ 1. **Data Collection and Preparation**
â”‚    - **Data**: Gather and prepare 3,701 Portuguese questions from ENEM,
â”‚ FUVEST, and Unicamp exams, ensuring that image-based questions are replaced
â”‚ with text descriptions.
â”‚    - **Tools**: Use Python libraries such as `pandas` for data manipulation,
â”‚ `requests` for web scraping, and `transformers` for model integration.
â”‚    - **Agent Pattern**: Develop a Python script that loads the selected
â”‚ small language model (Qwen3.5-4B) and preprocesses the data. The script
â”‚ should handle text preprocessing, tokenization, and data splitting for
â”‚ training and validation sets.
â”‚    - **Success Metric**: Track the accuracy of the model on the validation
â”‚ set to ensure the model is performing as expected.
â”‚ 
â”‚ 2. **Model Training and Evaluation

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 2.3 âœï¸ Questions

1. Count the LLM calls made by `tree_of_thoughts` with `k=3`. Compare with ReAct and Reflection.
   > *Your answer:* **2k + 1 = 7 calls** for `k = 3`:
   > - **k calls** to generate the ideas (`generate_thoughts`, one call per idea, each one seeing the previous ideas so that they differ);
   > - **k calls** to score each idea (`evaluate_thought`, structured output: feasibility + impact);
   > - **1 call** to develop the best idea into an action plan.
   >
   > Selecting the best idea (`max(...)`) is done by **code**, so it costs no LLM call.
   >
   > | Pattern | LLM calls | What drives the cost |
   > |---|---|---|
   > | **ReAct** | `n` (one per Thought/Action step, until the final answer) | Unknown in advance: depends on how many tool calls the model decides to make (bounded by `max_steps`) |
   > | **Reflection** | `2r + 1` worst case (1 draft + 1 critique and 1 revision per round) | Number of rounds before the critic approves (bounded by `max_rounds`) |
   > | **Tree of Thoughts** | `2k + 1` (depth 1) | **Fixed** by the breadth `k`: every branch is always generated and scored |
   >
   > ToT is the only one whose cost is fixed and paid upfront: it spends calls exploring alternatives even when the first idea was already good. That is the price of not committing to a single line of reasoning.
2. Our ToT has depth 1 (we develop only the best branch). How would you implement **depth 2**
   (develop the 2 best ideas, score again, keep the best)? Describe it in 3 lines.
   > *Your answer:*
   > 1. **Select the top 2 branches:** after scoring the `k` ideas, sort them by `feasibility + impact` and keep the 2 best instead of only the `max`.
   > 2. **Expand each branch:** develop both ideas into an action plan (2 calls), as `tree_of_thoughts` already does for one idea.
   > 3. **Score again and prune:** evaluate each developed plan with a second structured schema adapted to this deeper level (e.g. a `PlanEvaluation` Pydantic model with clarity of the steps, risks and measurability of the success metric) through an `evaluate_plan` function, and keep only the plan with the highest score.
   >
   > Cost: `k + k + 2 + 2 = 2k + 4` calls (10 for `k = 3`). Each extra level multiplies the work by the number of branches kept (the *beam width*), which is why ToT is usually implemented as a beam search with a small width.
3. For which of these tasks would you choose ReAct, Reflection, Plan-and-Execute or ToT?
   *(a) answer a factual question from a database Â· (b) write a cover letter Â· (c) organise a 3-day event Â· (d) choose a system architecture*
   > *Your answer:*
   > - **(a) Factual question from a database → ReAct.** The answer lives outside the model, so the agent must act (query the database), observe the result and possibly refine the query. A short think → act → observe loop is enough; planning or exploring alternatives would only add cost.
   > - **(b) Cover letter → Reflection.** There is no external data to fetch and no single correct answer: quality comes from iterating on the text. A critic checking tone, length and the match with the job offer, followed by revisions, is exactly the generate → critique → revise loop.
   > - **(c) 3-day event → Plan-and-Execute.** The goal decomposes into many ordered, dependent sub-tasks (budget → venue → schedule → catering → communication). Writing the global plan first prevents forgetting steps and lets each step reuse the previous results.
   > - **(d) System architecture → Tree of Thoughts.** Several valid options exist (monolith, microservices, serverless...) and the decision is costly to reverse. ToT generates alternatives, scores them against explicit criteria (cost, scalability, team skills) and develops the best one instead of committing to the first idea.
   >
   > The patterns can also be combined: e.g. Plan-and-Execute for the event where each step is executed by a ReAct agent with tools (booking, search), or a ToT whose final plan is polished by Reflection.

---
## Section 3 â€” Multi-agent systems with CrewAI

In Sections 1â€“2, **one** LLM played several roles inside **your** Python loop.
A **multi-agent framework** makes those roles first-class objects that collaborate.

| Concept | CrewAI object | Key fields | Question it answers |
|---------|---------------|-----------|---------------------|
| Agent | `Agent` | `role`, `goal`, `backstory`, `tools`, `llm` | **Who** does the work? |
| Task | `Task` | `description`, `expected_output`, `agent`, `context` | **What** must be produced? |
| Crew | `Crew` | `agents`, `tasks`, `process` | **How** do they collaborate? |
| Process | `Process.sequential` / `Process.hierarchical` | â€” | in order / via a manager agent |

**Why `role`, `goal` and `backstory`?** CrewAI injects them into the agent's system prompt.
This *persona conditioning* steers vocabulary, level of detail and behaviour (e.g. "you never invent facts").
Internally, each agent still runs a **ReAct loop** with its tools â€” exactly what you coded in TP1.

```
   inputs = {"topic": ...}
          â”‚
          â–¼
   â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”     search_report(query)     â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
   â”‚  RESEARCHER  â”‚ â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â–º â”‚ ChromaDB (TP2)   â”‚
   â”‚  ReAct loop  â”‚ â—„â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€ â”‚ your paper       â”‚
   â””â”€â”€â”€â”€â”€â”€â”¬â”€â”€â”€â”€â”€â”€â”€â”˜          passages            â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
          â”‚  research notes  (Task.context)
          â–¼
   â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
   â”‚    WRITER    â”‚ â”€â”€â–º paper_summary.md
   â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
```

### 3.1 â€” Load the knowledge base from TP2

In [9]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

if Path(CHROMA_DIR).exists():
    vectorstore = Chroma(
        collection_name="rapport_recursive",
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )
    print(f"âœ… TP2 index loaded â€” {vectorstore._collection.count()} chunks")
else:
    print("âš ï¸  chroma_db/ not found â€” building a small demo index (copy RAG/chroma_db here for the real thing)")
    SAMPLE = [
        ("Abstract. We present a benchmark for Small Language Models (SLMs) on Brazilian high-stakes "
         "exams (ENEM, FUVEST, Unicamp), with a zero-shot evaluation of 12 models.", 0),
        ("Methods. 3,701 Portuguese multiple-choice questions; visual items replaced by text descriptions "
         "generated by Seed-1.8; zero-shot prompts with strict JSON output, temperature 0.", 2),
        ("Results. GLM-5.1 led with 92.04% average accuracy. Among SLMs, Qwen3.5-4B reached 84.55%, "
         "beating Gemma-3-27B (76.73%). Qwen3.5-0.8B scored 42.44%.", 5),
        ("Limitations and future work. Text descriptions remove true spatial reasoning; third-party API "
         "non-determinism. Future work: fine-tuning for math and format adherence, native multimodal SLMs.", 9),
    ]
    docs = [Document(page_content=t, metadata={"page": p}) for t, p in SAMPLE]
    vectorstore = Chroma.from_documents(docs, embeddings, collection_name="demo_paper")
    print(f"âœ… Demo index built â€” {len(docs)} chunks")

âœ… TP2 index loaded â€” 84 chunks


### 3.2 â€” A CrewAI tool over the paper

> âš ï¸ CrewAI has **its own** `@tool` decorator (`crewai.tools.tool`). It is **not** the LangChain one from TP1/TP2 â€”
> we import it as `crew_tool` to avoid confusion. Same principle though: **the docstring is the tool description** the LLM reads.

In [10]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool as crew_tool


@crew_tool("search_report")
def search_report(query: str) -> str:
    """Search the student's research paper on Small Language Models evaluated on Brazilian entrance exams
    (ENEM, FUVEST, Unicamp) and return the 4 most relevant passages with their page numbers.
    Use short, specific queries such as 'research objectives', 'evaluation protocol', 'results accuracy'
    or 'limitations and future work'."""
    docs = vectorstore.similarity_search(query, k=4)
    if not docs:
        return "No passage found."
    return "\n\n".join(f"[page {d.metadata.get('page', '?')}] {d.page_content}" for d in docs)


# Test the tool directly, without any agent
print(search_report.run(query="main objectives of the paper")[:400])

[page 1] In summary, our main contributions are: (1) a comprehensive evaluation of mod-
ern SLMs on major Brazilian university entrance exams; (2) a granular analysis of model
performance mapped across specific disciplines and cognitive capabilities; (3) an empir-
ical assessment of the performance penalty incurred by replacing visual questions with
textual descriptions; and (4) a detailed cost-be


### 3.3 â€” Agents: role, goal, backstory

In [11]:
crew_llm = LLM(model=f"ollama/{CREW_MODEL}", base_url="http://localhost:11434", temperature=0)

researcher = Agent(
    role="Research Paper Analyst",
    goal="Extract accurate, sourced facts about {topic} from the research paper",
    backstory=(
        "You are a meticulous research assistant at IMT Mines Alès. "
        "You never invent facts: every claim you report comes from the search_report tool "
        "and is followed by its page number."
    ),
    tools=[search_report],
    llm=crew_llm,
    allow_delegation=False,
    max_iter=5,          # safety limit of the internal ReAct loop (cf. TP1 max_steps)
    verbose=True,
)

writer = Agent(
    role="Technical Writer",
    goal="Turn research notes about a scientific paper into a clear executive summary for a jury of engineers",
    backstory=(
        "You are a science communicator who writes for busy engineering managers. "
        "You use only the facts provided by the researcher and you never add numbers that are not in the notes."
    ),
    llm=crew_llm,
    allow_delegation=False,
    verbose=True,
)

print("âœ… Agents ready:", researcher.role, "|", writer.role)

âœ… Agents ready: Research Paper Analyst | Technical Writer


### 3.4 â€” Tasks and crew

`expected_output` is a **contract**: the more precise it is, the better small models follow it.
`context=[research_task]` passes the researcher's output to the writer.

In [12]:
research_task = Task(
    description=(
        "Search the research paper for information about: {topic}.\n"
        "Make 2 or 3 searches with different queries (objectives, methods, results, limitations)."
    ),
    expected_output=(
        "A bullet list of 6 to 10 facts, each followed by its page number, "
        "e.g. '- Qwen3.5-4B reached 84.55% average accuracy (p. 5)'."
    ),
    agent=researcher,
)

writing_task = Task(
    description="Write an executive summary of the research paper about {topic}, using ONLY the research notes.",
    expected_output=(
        "A markdown document: a title, then 3 short paragraphs (context, work done, results). Max 200 words."
    ),
    agent=writer,
    context=[research_task],
    output_file="paper_summary.md",
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

print("âœ… Crew ready")

âœ… Crew ready


### 3.5 â€” Run the crew

This takes 1â€“3 minutes on a laptop. Read the verbose logs: you will recognise the **Thought / Action / Observation**
structure of ReAct inside the researcher.

> âš ï¸ **Jupyter:** use `await crew.kickoff_async(...)`. Jupyter already runs an event loop, and recent CrewAI versions
> refuse the synchronous `crew.kickoff()` inside it (*"invoked synchronously from within a running event loop"*).
> In a plain `.py` script, `crew.kickoff(...)` is fine.

In [13]:
TOPIC = "the main contribution of the paper: objectives, methods and results"

result = await crew.kickoff_async(inputs={"topic": TOPIC})

show("FINAL OUTPUT (writer)", result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e1f214af-6b59-4ebb-96cc-e2cd7f043885                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the research paper for information about: the main contribution of the paper: objectives,         │
│  methods and results.                                                                                           │
│  Make 2 or 3 searches with different queries (objectives, methods, results, limitations).                       │
│  ID: 04494704-f7a1-432a-b36a-d9a5c50b3598                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Paper Analyst                                                                                  │
│                                                                                                                 │
│  Task: Search the research paper for information about: the main contribution of the paper: objectives,         │
│  methods and results.                                                                                           │
│  Make 2 or 3 searches with different queries (objectives, methods, results, limitations).                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Using an approximate input token count for model 'qwen2.5:3b': assuming one token per four characters because a native token count is unavailable.
The context window of model 'qwen2.5:3b' is unknown, so CrewAI assumes 8,192 tokens (6,963 usable). If the model's window is different, pass it as LLM(model=..., context_window_size=<tokens>).


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution objectives'}                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution methods'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (10 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Executive Summary                                                                                            │
│                                                                                                                 │
│  This research paper evaluates modern Small Language Models (SLMs) on major Brazilian university entrance       │
│  exams, including ENEM, FUVEST (USP), and Unicamp. The study replaces visual questions with textual             │
│  descriptions to isolate reasoning and uses a zero-shot prompting protocol to elicit strict JSON responses.     │
│  The main contributions are:                                                                                    │
│                                                                                                                 │
│  1. **Comprehensive Evaluation**: The paper evaluates SLMs on major Brazilian university entrance exams,        │
│  providing a granular analysis of model performance across specific disciplines and cognitive capabilities.     │
│  2. **Performance Penalty Analysis**: The study assesses the performance penalty incurred by replacing visual   │
│  questions with textual descriptions, highlighting the impact on model accuracy.                                │
│  3. **Cost-Benefit Analysis**: A detailed cost-benefit analysis delineates the Pareto frontier, offering        │
│  insights into the trade-offs between model size and performance.                                               │
│                                                                                                                 │
│  The evaluation results show that Qwen3.5-4B achieved an accuracy of 86.08% across all subjects and years       │
│  tested (2018-2024). This comprehensive approach, including the adaptation of visual assets to text             │
│  descriptions, effectively closes the performance gap with larger models, demonstrating the potential of SLMs   │
│  in evaluating university entrance exams.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an executive summary of the research paper about the main contribution of the paper: objectives,   │
│  methods and results, using ONLY the research notes.                                                            │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e1f214af-6b59-4ebb-96cc-e2cd7f043885                                                                       │
│  Final Output: # Executive Summary                                                                              │
│                                                                                                                 │
│  This research paper evaluates modern Small Language Models (SLMs) on major Brazilian university entrance       │
│  exams, including ENEM, FUVEST (USP), and Unicamp. The study replaces visual questions with textual             │
│  descriptions to isolate reasoning and uses a zero-shot prompting protocol to elicit strict JSON responses.     │
│  The main contributions are:                                                                                    │
│                                                                                                                 │
│  1. **Comprehensive Evaluation**: The paper evaluates SLMs on major Brazilian university entrance exams,        │
│  providing a granular analysis of model performance across specific disciplines and cognitive capabilities.     │
│  2. **Performance Penalty Analysis**: The study assesses the performance penalty incurred by replacing visual   │
│  questions with textual descriptions, highlighting the impact on model accuracy.                                │
│  3. **Cost-Benefit Analysis**: A detailed cost-benefit analysis delineates the Pareto frontier, offering        │
│  insights into the trade-offs between model size and performance.                                               │
│                                                                                                                 │
│  The evaluation results show that Qwen3.5-4B achieved an accuracy of 86.08% across all subjects and years       │
│  tested (2018-2024). This comprehensive approach, including the adaptation of visual assets to text             │
│  descriptions, effectively closes the performance gap with larger models, demonstrating the potential of SLMs   │
│  in evaluating university entrance exams.                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


â”Œâ”€ FINAL OUTPUT (writer) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ # Executive Summary
â”‚ 
â”‚ This research paper evaluates modern Small Language Models (SLMs) on major
â”‚ Brazilian university entrance exams, including ENEM, FUVEST (USP), and
â”‚ Unicamp. The study replaces visual questions with textual descriptions to
â”‚ isolate reasoning and uses a zero-shot prompting protocol to elicit strict
â”‚ JSON responses. The main contributions are:
â”‚ 
â”‚ 1. **Comprehensive Evaluation**: The paper evaluates SLMs on major Brazilian
â”‚ university entrance exams, providing a granular analysis of model
â”‚ performance across specific disciplines and cognitive capabilities.
â”‚ 2. **Performance Penalty Analysis**: The study assesses the performance
â”‚ penalty incurred by replacing visual questions with textual descriptions,
â”‚ highlighting the impact on model acc

In [14]:
# What did each agent produce?
for task_output in result.tasks_output:
    show(f"{task_output.agent}", task_output.raw[:800])

print("\nToken usage:", result.token_usage)
print("Summary saved to: paper_summary.md")


â”Œâ”€ Research Paper Analyst â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ - In summary, our main contributions are: (1) a comprehensive evaluation of
â”‚ modern Small Language Models (SLMs) on major Brazilian university entrance
â”‚ exams; (2) a granular analysis of model performance mapped across specific
â”‚ disciplines and cognitive capabilities; (3) an empirical assessment of the
â”‚ performance penalty incurred by replacing visual questions with textual
â”‚ descriptions; and (4) a detailed cost-benefit analysis delineating the
â”‚ Pareto frontier (p. 1).
â”‚ - The dataset comprises questions from ENEM, FUVEST (USP), and Unicamp, with
â”‚ visual assets replaced by text descriptions to isolate reasoning and using a
â”‚ zero-shot prompting protocol to elicit strict JSON responses (p. 1).
â”‚ - The evaluation protocol includes a comprehensive evaluation of SLMs on
â”‚ 

### 3.6 âœï¸ Exercise A â€” Add a reviewer agent (reflection, multi-agent version)

Section 1's critic becomes an **agent**. Create a `reviewer` that compares the summary with the research notes,
**flags every claim not supported by the notes**, and returns a corrected version.

Steps:
1. Create `reviewer = Agent(...)` with a role, a goal and a backstory of your choice (no tools needed).
2. Create `review_task = Task(...)` with `context=[research_task, writing_task]` and a precise `expected_output`
   (e.g. *"A list of unsupported claims, then the corrected summary"*).
3. Build `crew_v2` with the 3 agents and 3 tasks, and run it.

In [15]:
# ── Exercise A: reviewer agent (reflection, multi-agent version) ────────────
# 1. The reviewer plays the role of Section 1's critic, but as a CrewAI agent.
#    No tools: it only compares two texts it receives through Task.context.
reviewer = Agent(
    role="Scientific Fact-Checker",
    goal="Make sure every claim of the executive summary is supported by the research notes",
    backstory=(
        "You are a rigorous reviewer for a scientific journal. "
        "You compare the summary with the research notes sentence by sentence. "
        "A claim is supported only if the notes state it explicitly: same number, same model name, same page. "
        "You never add new facts; when a claim is unsupported, you remove it or rewrite it with what the notes say."
    ),
    llm=crew_llm,
    allow_delegation=False,
    verbose=True,
)

# 2. context=[research_task, writing_task]: CrewAI injects BOTH previous outputs
#    (the notes and the summary) into the reviewer's prompt.
review_task = Task(
    description=(
        "Review the executive summary about {topic} against the research notes.\n"
        "For each sentence of the summary, check whether the notes support it. "
        "Flag every claim (number, model name, conclusion) that is NOT in the notes, "
        "then rewrite the summary using only supported facts."
    ),
    expected_output=(
        "A markdown document with exactly two sections:\n"
        "## Unsupported claims\n"
        "A bullet list of each unsupported claim and why (or '- None' if every claim is supported).\n"
        "## Corrected summary\n"
        "The corrected summary: a title, then 3 short paragraphs (context, work done, results). Max 200 words."
    ),
    agent=reviewer,
    context=[research_task, writing_task],
    output_file="paper_summary_reviewed.md",
)

# 3. Same sequential process, now with 3 agents and 3 tasks:
#    researcher → writer → reviewer
crew_v2 = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, writing_task, review_task],
    process=Process.sequential,
    verbose=True,
)

result_v2 = await crew_v2.kickoff_async(inputs={"topic": TOPIC})

# result_v2.tasks_output keeps the output of each task, in order.
notes_v2, summary_v2, review_v2 = (t.raw for t in result_v2.tasks_output)
show("RESEARCH NOTES (researcher)", notes_v2)
show("SUMMARY BEFORE REVIEW (writer)", summary_v2)
show("REVIEWED SUMMARY (reviewer)", review_v2)
print("\nToken usage:", result_v2.token_usage)
print("Reviewed summary saved to: paper_summary_reviewed.md")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e4ce5faa-5a87-4ac2-badb-1733b4195a40                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the research paper for information about: the main contribution of the paper: objectives,         │
│  methods and results.                                                                                           │
│  Make 2 or 3 searches with different queries (objectives, methods, results, limitations).                       │
│  ID: 04494704-f7a1-432a-b36a-d9a5c50b3598                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Paper Analyst                                                                                  │
│                                                                                                                 │
│  Task: Search the research paper for information about: the main contribution of the paper: objectives,         │
│  methods and results.                                                                                           │
│  Make 2 or 3 searches with different queries (objectives, methods, results, limitations).                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution objectives'}                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution methods'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution results'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (14 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scientific Fact-Checker                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Unsupported claims                                                                                          │
│  - None                                                                                                         │
│                                                                                                                 │
│  ## Corrected summary                                                                                           │
│  This research paper evaluates modern Small Language Models (SLMs) on major Brazilian university entrance       │
│  exams, including ENEM, FUVEST (USP), and Unicamp. The study replaces visual questions with textual             │
│  descriptions to isolate reasoning and uses a zero-shot prompting protocol to elicit strict JSON responses.     │
│  The main contributions are:                                                                                    │
│  1. **Comprehensive Evaluation**: The paper evaluates SLMs on major Brazilian university entrance exams,        │
│  providing a granular analysis of model performance across specific disciplines and cognitive capabilities.     │
│  2. **Performance Penalty Analysis**: The study assesses the performance penalty incurred by replacing visual   │
│  questions with textual descriptions, highlighting the impact on model accuracy.                                │
│  3. **Cost-Benefit Analysis**: A detailed cost-benefit analysis delineates the Pareto frontier, offering        │
│  insights into the trade-offs between model size and performance.                                               │
│                                                                                                                 │
│  The evaluation results show that Qwen3.5-4B achieved an accuracy of 86.08% across all subjects and years       │
│  tested (2018-2024). This comprehensive approach, including the adaptation of visual assets to text             │
│  descriptions, effectively closes the performance gap with larger models, demonstrating the potential of SLMs   │
│  in evaluating university entrance exams.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the executive summary about the main contribution of the paper: objectives, methods and results   │
│  against the research notes.                                                                                    │
│  For each sentence of the summary, check whether the notes support it. Flag every claim (number, model name,    │
│  conclusion) that is NOT in the notes, then rewrite the summary using only supported facts.                     │
│  Agent: Scientific Fact-Checker                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e4ce5faa-5a87-4ac2-badb-1733b4195a40                                                                       │
│  Final Output: ## Unsupported claims                                                                            │
│  - None                                                                                                         │
│                                                                                                                 │
│  ## Corrected summary                                                                                           │
│  This research paper evaluates modern Small Language Models (SLMs) on major Brazilian university entrance       │
│  exams, including ENEM, FUVEST (USP), and Unicamp. The study replaces visual questions with textual             │
│  descriptions to isolate reasoning and uses a zero-shot prompting protocol to elicit strict JSON responses.     │
│  The main contributions are:                                                                                    │
│  1. **Comprehensive Evaluation**: The paper evaluates SLMs on major Brazilian university entrance exams,        │
│  providing a granular analysis of model performance across specific disciplines and cognitive capabilities.     │
│  2. **Performance Penalty Analysis**: The study assesses the performance penalty incurred by replacing visual   │
│  questions with textual descriptions, highlighting the impact on model accuracy.                                │
│  3. **Cost-Benefit Analysis**: A detailed cost-benefit analysis delineates the Pareto frontier, offering        │
│  insights into the trade-offs between model size and performance.                                               │
│                                                                                                                 │
│  The evaluation results show that Qwen3.5-4B achieved an accuracy of 86.08% across all subjects and years       │
│  tested (2018-2024). This comprehensive approach, including the adaptation of visual assets to text             │
│  descriptions, effectively closes the performance gap with larger models, demonstrating the potential of SLMs   │
│  in evaluating university entrance exams.                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


â”Œâ”€ RESEARCH NOTES (researcher) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ - In summary, our main contributions are: (1) a comprehensive evaluation of
â”‚ modern Small Language Models (SLMs) on major Brazilian university entrance
â”‚ exams; (2) a granular analysis of model performance mapped across specific
â”‚ disciplines and cognitive capabilities; (3) an empirical assessment of the
â”‚ performance penalty incurred by replacing visual questions with textual
â”‚ descriptions; and (4) a detailed cost-benefit analysis delineating the
â”‚ Pareto frontier (p. 1).
â”‚ - The dataset comprises questions from ENEM, FUVEST (USP), and Unicamp, with
â”‚ visual assets replaced by text descriptions to isolate reasoning and using a
â”‚ zero-shot prompting protocol to elicit strict JSON responses (p. 1).
â”‚ - The evaluation protocol includes a comprehensive evaluation of SLMs on
â”‚ major Braz

### 3.7 âœï¸ Exercise B â€” Does the backstory matter?

Re-create the `writer` with a very different backstory, for example:
*"You are an enthusiastic marketing intern who loves superlatives and emojis."*
Run the crew again with the same topic and compare the two summaries.

In [16]:
# ── Exercise B: does the backstory matter? ──────────────────────────────────
import re

# Only the backstory changes: same role, same goal, same LLM, same task.
# Any difference in the output can therefore be attributed to the backstory.
writer_marketing = Agent(
    role="Technical Writer",
    goal=writer.goal,
    backstory="You are an enthusiastic marketing intern who loves superlatives and emojis.",
    llm=crew_llm,
    allow_delegation=False,
    verbose=True,
)

writing_task_marketing = Task(
    description=writing_task.description,
    expected_output=writing_task.expected_output,
    agent=writer_marketing,
    context=[research_task],
    output_file="paper_summary_marketing.md",
)

crew_marketing = Crew(
    agents=[researcher, writer_marketing],
    tasks=[research_task, writing_task_marketing],
    process=Process.sequential,
    verbose=True,
)

result_marketing = await crew_marketing.kickoff_async(inputs={"topic": TOPIC})
notes_marketing = result_marketing.tasks_output[0].raw


def unsupported_numbers(summary: str, notes: str) -> list[str]:
    """Return the numbers of a summary that do not appear in the research notes.

    A cheap, deterministic fact check done by code (cf. Section 1.4): any number
    the writer used but the researcher never reported is a likely invention.

    Args:
        summary: The text produced by the writer.
        notes: The research notes the writer was given.

    Returns:
        The sorted list of numbers (as strings, e.g. "84.55") found in
        ``summary`` but not in ``notes``.
    """
    number = r"\d+(?:[.,]\d+)?"
    in_notes = set(re.findall(number, notes))
    return sorted(set(re.findall(number, summary)) - in_notes)


# Side-by-side comparison: original writer (Section 3.5) vs marketing writer
for label, output, notes in [
    ("ORIGINAL WRITER (science communicator)", result.raw, result.tasks_output[0].raw),
    ("MARKETING WRITER (enthusiastic intern)", result_marketing.raw, notes_marketing),
]:
    show(label, output)
    print(f"  words: {len(output.split())} | "
          f"emojis: {count_emojis(output)} | "
          f"numbers NOT in its research notes: {unsupported_numbers(output, notes) or 'none'}")

print("\nSame research notes in both runs?", notes_marketing == result.tasks_output[0].raw)
print("Marketing summary saved to: paper_summary_marketing.md")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2754e908-7738-41d4-b19b-38582c0ecdbb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the research paper for information about: the main contribution of the paper: objectives,         │
│  methods and results.                                                                                           │
│  Make 2 or 3 searches with different queries (objectives, methods, results, limitations).                       │
│  ID: 04494704-f7a1-432a-b36a-d9a5c50b3598                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Paper Analyst                                                                                  │
│                                                                                                                 │
│  Task: Search the research paper for information about: the main contribution of the paper: objectives,         │
│  methods and results.                                                                                           │
│  Make 2 or 3 searches with different queries (objectives, methods, results, limitations).                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#7) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution objectives'}                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution methods'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#9) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_report                                                                                            │
│  Args: {'query': 'main contribution results'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (9 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Executive Summary                                                                                            │
│                                                                                                                 │
│  Our research evaluates modern Small Language Models (SLMs) on major Brazilian university entrance exams,       │
│  including ENEM, FUVEST (USP), and Unicamp. We replaced visual assets with text descriptions to isolate         │
│  reasoning and used a zero-shot prompting protocol for strict JSON responses (p. 1).                            │
│                                                                                                                 │
│  Qwen3.5-4B achieved an accuracy of 86.08% across all subjects tested, including FUVEST and Unicamp subjects    │
│  from 2018 to 2024 (p. 6, 8). This model demonstrated strong performance, surpassing expectations and closing   │
│  the performance gap with larger models (p. 1). Our granular analysis reveals that Qwen3.5-4B excels in         │
│  specific disciplines and cognitive capabilities, with a detailed cost-benefit analysis delineating the Pareto  │
│  frontier (p. 1).                                                                                               │
│                                                                                                                 │
│  Qwen3.5-4B's performance penalty for visual questions was minimal, with no significant drop in accuracy when   │
│  textual descriptions were used (p. 6, 8). This study provides valuable insights into the capabilities and      │
│  limitations of SLMs in evaluating university entrance exams, offering a comprehensive evaluation and           │
│  empirical assessment (p. 1).                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an executive summary of the research paper about the main contribution of the paper: objectives,   │
│  methods and results, using ONLY the research notes.                                                            │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2754e908-7738-41d4-b19b-38582c0ecdbb                                                                       │
│  Final Output: # Executive Summary                                                                              │
│                                                                                                                 │
│  Our research evaluates modern Small Language Models (SLMs) on major Brazilian university entrance exams,       │
│  including ENEM, FUVEST (USP), and Unicamp. We replaced visual assets with text descriptions to isolate         │
│  reasoning and used a zero-shot prompting protocol for strict JSON responses (p. 1).                            │
│                                                                                                                 │
│  Qwen3.5-4B achieved an accuracy of 86.08% across all subjects tested, including FUVEST and Unicamp subjects    │
│  from 2018 to 2024 (p. 6, 8). This model demonstrated strong performance, surpassing expectations and closing   │
│  the performance gap with larger models (p. 1). Our granular analysis reveals that Qwen3.5-4B excels in         │
│  specific disciplines and cognitive capabilities, with a detailed cost-benefit analysis delineating the Pareto  │
│  frontier (p. 1).                                                                                               │
│                                                                                                                 │
│  Qwen3.5-4B's performance penalty for visual questions was minimal, with no significant drop in accuracy when   │
│  textual descriptions were used (p. 6, 8). This study provides valuable insights into the capabilities and      │
│  limitations of SLMs in evaluating university entrance exams, offering a comprehensive evaluation and           │
│  empirical assessment (p. 1).                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


â”Œâ”€ ORIGINAL WRITER (science communicator) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
â”‚ # Executive Summary
â”‚ 
â”‚ This research paper evaluates modern Small Language Models (SLMs) on major
â”‚ Brazilian university entrance exams, including ENEM, FUVEST (USP), and
â”‚ Unicamp. The study replaces visual questions with textual descriptions to
â”‚ isolate reasoning and uses a zero-shot prompting protocol to elicit strict
â”‚ JSON responses. The main contributions are:
â”‚ 
â”‚ 1. **Comprehensive Evaluation**: The paper evaluates SLMs on major Brazilian
â”‚ university entrance exams, providing a granular analysis of model
â”‚ performance across specific disciplines and cognitive capabilities.
â”‚ 2. **Performance Penalty Analysis**: The study assesses the performance
â”‚ penalty incurred by replacing visual questions with textual descriptions,
â”‚ highlighting the impact on model accuracy.
â”‚ 3. **Cost-Benefit Analy

### 3.8 âœï¸ Questions

1. Compare the two summaries of Exercise B. What changed? Did the facts change, or only the style?
   > *Your answer:* **The style changed; the numbers did not; but some facts did change.** Both runs used exactly the same research notes (`Same research notes in both runs? True`), so the comparison is fair.
   > - **Style:** the marketing writer switched to the first person ("Our research", "We replaced"), followed the "3 paragraphs" format better than the original writer (which used a numbered list), added inline page citations and enthusiastic wording ("surpassing expectations", "excels", "valuable insights"). Surprisingly, it used **0 emojis** and few superlatives: for a 3B model, the task description ("using ONLY the research notes") and the `expected_output` weighed more than the backstory.
   > - **Numbers:** identical. The code check found no number outside the notes in either summary (both repeat the 86.08% of the notes).
   > - **Facts:** the marketing writer **invented qualitative claims** absent from the notes: "Qwen3.5-4B excels in specific disciplines" and "the performance penalty for visual questions was minimal, with no significant drop in accuracy". The last one is the **opposite** of the paper, which reports a clear penalty for small models. The numeric check could not catch it: a number-based guardrail does not cover qualitative claims.
   >
   > So the backstory mostly changes the tone, but a "promotional" persona also makes the model more willing to embellish conclusions.
2. With `qwen2.5:3b`, what failures did you observe (tool not called, wrong format, invented page numbers, loops)?
   Which ones did `max_iter`, `expected_output` or the backstory help with?
   > *Your answer:* Observed failures (same behaviour in the 3 runs, since `temperature=0` and the researcher always issued the same 3 queries):
   > - **Tool not called:** did not happen. The researcher made 3 searches with different queries (objectives, methods, results), as requested.
   > - **Invented numbers and page numbers:** the key fact of the notes, *"Qwen3.5-4B achieved 86.08%"*, does not exist in the paper (the real average is 84.55%). The model misread a garbled heatmap chunk (`86.085.184.6...` = 86.0, 85.1, 84.6 glued together by the PDF extraction) and attributed the same number to p. 8, a chunk that contains only figure labels and no number at all.
   > - **Wrong format / repetition loop:** 13 bullets instead of 6–10, most of them near-duplicates ("...also show that Qwen3.5-4B achieved 86.08%..."), a typical degeneration of small models. The writer also ignored "3 short paragraphs" and produced a numbered list.
   > - **Error propagation and over-generalisation:** the writer turned "86.08% for FUVEST subjects" into "86.08% across all subjects and years" and wrote that the approach "effectively closes the performance gap with larger models", which the paper does not say.
   > - **Lenient reviewer:** the reviewer answered "Unsupported claims: None" and copied the summary verbatim (Exercise A). It is the "too lenient" critic of Section 1, and it cannot fix errors that are already in the notes (garbage in, garbage out).
   >
   > What helped: **`max_iter`** bounded the researcher's ReAct loop (it never looped on tool calls); **`expected_output`** fixed the *format* (bullets with page numbers, the reviewer's two exact sections); the **backstory** ("every claim is followed by its page number") made the agent *cite* pages. None of them guaranteed *truthfulness*: format compliance is not factual accuracy. Fixing that requires better data (clean table extraction) and checks done by code.
3. CrewAI vs your hand-written loops of Sections 1â€“2: what did you **gain** and what did you **lose** in control?
   > *Your answer:*
   > - **Gained:** a declarative, readable design (roles, goals, tasks); the ReAct loop with tool calling, the passing of outputs between agents (`context`), output files, logs and token usage for free; adding a third agent took a few lines; switching to another process (hierarchical) is one parameter.
   > - **Lost:** control of the **prompts** (CrewAI builds hidden system prompts from role/goal/backstory) and of the **control flow**: the agent decides when it is done, while in Sections 1–2 the stopping rule was **code** (`score >= threshold`, word count). We cannot easily insert deterministic checks inside the loop, the tool calls are hidden (the 3 searches were sent at once), the cost grows silently (the v2 crew reported 21 requests and ~30k tokens) and debugging is harder (opaque warnings, a context window assumed at 8,192 tokens).
   >
   > Part of the control can be recovered with CrewAI features such as structured task outputs (`output_pydantic`) and task guardrails, or by moving to an explicit graph, which is the goal of LangGraph in TP4.
4. ðŸŽ *Bonus:* with `Process.hierarchical` and `manager_llm=crew_llm`, a manager agent assigns tasks itself.
   Why is this risky with a 3B model?
   > *Your answer:* In hierarchical mode, the 3B model becomes the **manager**: it must plan, delegate each task to the right coworker through tool calls (with the exact role name), judge the returned work and decide when the job is finished. These are the hardest skills for a small model, and we already saw it fail at easier ones (invented numbers, repetitions, a reviewer that approves everything). Typical risks: delegating to a wrong or non-existent coworker, delegating everything to one agent or doing the work itself, looping on delegations, accepting bad outputs, and exceeding the context window as the delegation history grows. In sequential mode the order is fixed by our code; in hierarchical mode the control flow itself depends on the weakest component, so errors compound and the run becomes unpredictable and more expensive.

---
## Checkpoint â€” What you have built

```
  Pattern               Implementation                       Status
  â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€ â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€ â”€â”€â”€â”€â”€â”€
  Reflection            generate â†’ critique â†’ revise loop     âœ…
                        structured critic (score + issues)    âœï¸
  Planning              Plan-and-Execute                      âœ…
                        Tree of Thoughts (k ideas â†’ best)     âœ…
  Multi-agent           CrewAI researcher + writer            âœ…
                        + reviewer agent                      âœï¸
```

### Next â€” TP4 (second hour)

CrewAI hides the control flow. In TP4 you take it back with **LangGraph**: an explicit graph of nodes and edges,
a persistent state, conditional routing, and a **human-in-the-loop** that can pause the agent before a risky action.